In [0]:
from pyspark.sql import functions as F

# 1. Load source (Silver)
src = spark.table("retailnew.silver.customers_clean")

# 2. Create temp view for SQL MERGE
src.createOrReplaceTempView("src_customers")

# 3. Run SCD2 MERGE using SQL
spark.sql("""
MERGE INTO retailnew.gold.dim_customer AS tgt
USING src_customers AS src
ON tgt.customer_id = src.customer_id AND tgt.current_flag = true

WHEN MATCHED AND (
    COALESCE(tgt.first_name,'') <> COALESCE(src.first_name,'') OR
    COALESCE(tgt.last_name,'') <> COALESCE(src.last_name,'') OR
    COALESCE(tgt.email,'') <> COALESCE(src.email,'') OR
    COALESCE(tgt.gender,'') <> COALESCE(src.gender,'') OR
    COALESCE(tgt.city,'') <> COALESCE(src.city,'') OR
    COALESCE(tgt.state,'') <> COALESCE(src.state,'') OR
    COALESCE(tgt.country,'') <> COALESCE(src.country,'')
)
THEN UPDATE SET
    tgt.end_date = current_date(),
    tgt.current_flag = false

WHEN NOT MATCHED THEN INSERT (
    customer_id,
    first_name,
    last_name,
    email,
    gender,
    city,
    state,
    country,
    start_date,
    end_date,
    current_flag
)
VALUES (
    src.customer_id,
    src.first_name,
    src.last_name,
    src.email,
    src.gender,
    src.city,
    src.state,
    src.country,
    current_date(),
    null,
    true
)
""")
